# Frankenstein MPC notebook

This notebook merges the **building RC model**, the **heat pump model**, and the **thermal storage tank model** into a single control-oriented workflow.

## Current modelling choices
- **Building**: the original **3R2C** structure is preserved as much as possible.
- **Heat pump**: quasi-steady control-oriented model with **capacity** and **COP** maps.
- **Tank**: **3-node** stratified storage model.
- **Architecture**: **HP → Tank → Building**.
- **Control**: **economic MPC** with:
  - indoor comfort constraints,
  - slack variables for feasibility,
  - electricity cost based on Italian **F1/F2/F3** time bands,
  - small smoothing penalties on control actions.

## Notes
This is a **robust v1**. The goal is to provide a readable and extendable notebook, not the most detailed plant model possible at this stage.


In [ ]:
import importlib
import subprocess
import sys

required_packages = ["casadi"]
for package in required_packages:
    if importlib.util.find_spec(package) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", package, "-q"])


In [ ]:
from dataclasses import dataclass
from pathlib import Path

import casadi as ca
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.linalg import expm


## Configuration

The baseline sizing used here is:
- nominal heat pump thermal capacity: **10 kW**
- tank size: **25 L/kW**
- resulting tank volume: **250 L**

The electricity prices are represented with fixed year-round values by time band:
- **F1 = 0.3291 €/kWh**
- **F2 = 0.3439 €/kWh**
- **F3 = 0.3060 €/kWh**

The notebook automatically uses `data.csv` if it exists in the repository root.  
If no external dataset is found, a synthetic weather profile is generated so that the notebook remains runnable.


In [ ]:
# -----------------------------
# Global configuration
# -----------------------------
DT_HOURS = 0.25                    # 15 minutes
SIMULATION_HOURS = 72              # closed-loop simulation length
PREDICTION_HOURS = 6               # MPC horizon
HORIZON_STEPS = int(PREDICTION_HOURS / DT_HOURS)

PRICE_BY_BAND_EUR_PER_KWH = {
    "F1": 0.3291,
    "F2": 0.3439,
    "F3": 0.3060,
}

# Building RC model parameters (preserved from build_RC_MPC as much as possible)
BUILDING_PARAMS = {
    "Rea": 58.4895,   # [°C/kW]
    "Rie": 1.1521,    # [°C/kW]
    "Rinf": 15.748,   # [°C/kW]
    "Ce": 3.5,        # [kWh/°C]
    "Ci": 1.15,       # [kWh/°C]
    "Ai": 1.56,       # [m²] effective solar aperture
    "Ae": 0.122,      # [m²] effective solar aperture
    "heating_efficiency": 1.0,
}

# Heat pump baseline
P_HP_NOMINAL_KW = 10.0
TANK_VOLUME_M3 = 25.0 / 1000.0 * P_HP_NOMINAL_KW  # 25 L/kW

# Control penalties
TRACK_WEIGHT = 1.0
SLACK_WEIGHT = 1000.0
SMOOTH_WEIGHT = 0.02

# Comfort band
TI_MIN_C = 20.0
TI_MAX_C = 24.0
TI_REF_C = 22.0


In [ ]:
# -----------------------------
# Utilities
# -----------------------------
def discretize_linear_system(A: np.ndarray, B: np.ndarray, dt_hours: float) -> tuple[np.ndarray, np.ndarray]:
    """Discretize a continuous-time linear state-space model."""
    A = np.asarray(A, dtype=float)
    B = np.asarray(B, dtype=float)
    A_d = expm(A * dt_hours)
    B_d = np.linalg.solve(A, (A_d - np.eye(A.shape[0]))) @ B
    return A_d, B_d


def get_arera_time_band(timestamp: pd.Timestamp) -> str:
    """Return F1/F2/F3 using the standard Italian calendar-based time bands."""
    weekday = timestamp.weekday()  # Monday = 0
    hour = timestamp.hour + timestamp.minute / 60.0

    # Public holidays are ignored in this v1 and can be added later.
    if weekday <= 4:  # Monday-Friday
        if 8 <= hour < 19:
            return "F1"
        if (7 <= hour < 8) or (19 <= hour < 23):
            return "F2"
        return "F3"

    if weekday == 5:  # Saturday
        return "F2" if 7 <= hour < 23 else "F3"

    return "F3"  # Sunday


def load_or_generate_weather_data(
    dt_hours: float = DT_HOURS,
    simulation_hours: float = SIMULATION_HOURS,
    csv_path: str | Path = "data.csv",
) -> pd.DataFrame:
    """
    Load external inputs from data.csv if available.
    Required columns when reading an external file:
    - Ta : outdoor air temperature [°C]
    - Gv : global horizontal or effective solar signal [kW/m² equivalent input used by the RC model]

    Optional:
    - timestamp : datetime-like column
    """
    csv_path = Path(csv_path)

    if csv_path.exists():
        df = pd.read_csv(csv_path)

        if "Ta" not in df.columns or "Gv" not in df.columns:
            raise ValueError("The external dataset must contain at least 'Ta' and 'Gv' columns.")

        n_steps = int(simulation_hours / dt_hours) + HORIZON_STEPS + 1
        df = df.iloc[:n_steps].copy().reset_index(drop=True)

        if "timestamp" in df.columns:
            df["timestamp"] = pd.to_datetime(df["timestamp"])
        else:
            df["timestamp"] = pd.date_range(
                "2023-01-02 00:00:00",
                periods=len(df),
                freq=f"{int(dt_hours * 60)}min",
            )

        df["Ta"] = df["Ta"].astype(float)
        df["Gv"] = df["Gv"].astype(float)
        source = "external data.csv"
    else:
        n_steps = int(simulation_hours / dt_hours) + HORIZON_STEPS + 1
        t_hours = np.arange(n_steps) * dt_hours

        df = pd.DataFrame({
            "timestamp": pd.date_range(
                "2023-01-02 00:00:00",
                periods=n_steps,
                freq=f"{int(dt_hours * 60)}min",
            )
        })

        # Synthetic but smooth profile for a winter-like test
        df["Ta"] = 6.0 + 4.0 * np.sin(2.0 * np.pi * (t_hours - 8.0) / 24.0)
        df["Gv"] = np.maximum(0.0, 0.6 * np.sin(2.0 * np.pi * (t_hours - 6.0) / 24.0))
        source = "synthetic fallback profile"

    df["band"] = df["timestamp"].apply(get_arera_time_band)
    df["price_eur_per_kwh"] = df["band"].map(PRICE_BY_BAND_EUR_PER_KWH)

    print(f"Weather / price source: {source}")
    print(f"Rows available: {len(df)}")
    return df


df_inputs = load_or_generate_weather_data()
df_inputs.head()


## Building model

The building model keeps the original **3R2C** formulation and its parameter set.  
The only conceptual change is that the thermal input is now interpreted as:

\[
Q_{build}
\]

that is, the **thermal power effectively delivered to the building from the tank**, not directly from the heat pump.


In [ ]:
# -----------------------------
# Building RC model
# -----------------------------
@dataclass
class BuildingRCParameters:
    Rea: float
    Rie: float
    Rinf: float
    Ce: float
    Ci: float
    Ai: float
    Ae: float
    heating_efficiency: float = 1.0


class BuildingRCModel:
    def __init__(self, params: BuildingRCParameters, dt_hours: float) -> None:
        self.params = params
        self.dt_hours = dt_hours

        A_c = np.array([
            [
                -1.0 / (params.Ci * params.Rie) - 1.0 / (params.Ci * params.Rinf),
                1.0 / (params.Ci * params.Rie),
            ],
            [
                1.0 / (params.Ce * params.Rie),
                -1.0 / (params.Ce * params.Rie) - 1.0 / (params.Ce * params.Rea),
            ],
        ])

        B_c = np.array([
            [1.0 / params.Ci, 1.0 / (params.Ci * params.Rinf), params.Ai / params.Ci],
            [0.0, 1.0 / (params.Ce * params.Rea), params.Ae / params.Ce],
        ])

        self.A_d, self.B_d = discretize_linear_system(A_c, B_c, dt_hours)

    def step_numpy(self, x_k: np.ndarray, q_build_kw: float, t_out_c: float, g_solar: float) -> np.ndarray:
        u_k = np.array([
            q_build_kw * self.params.heating_efficiency,
            t_out_c,
            g_solar,
        ])
        return self.A_d @ x_k + self.B_d @ u_k


building_model = BuildingRCModel(
    BuildingRCParameters(**BUILDING_PARAMS),
    dt_hours=DT_HOURS,
)

print("Discrete A matrix:")
print(np.round(building_model.A_d, 4))
print("\nDiscrete B matrix:")
print(np.round(building_model.B_d, 4))


## Heat pump model

The heat pump is modeled in a quasi-steady way through:
- a **capacity map**
- a **COP map**

The sink-side temperature is approximated by the **top tank node temperature**.  
This makes the MPC naturally sensitive to the storage state.


In [ ]:
# -----------------------------
# Heat pump model
# -----------------------------
@dataclass
class QuasiSteadyHeatPumpParameters:
    q_ref_kw: float
    cop_ref: float
    t_source_ref_c: float
    t_sink_ref_c: float
    alpha_q_src: float
    alpha_q_sink: float
    alpha_cop_src: float
    alpha_cop_sink: float


HP_PARAMS = QuasiSteadyHeatPumpParameters(
    q_ref_kw=P_HP_NOMINAL_KW,
    cop_ref=3.5,
    t_source_ref_c=7.0,
    t_sink_ref_c=35.0,
    alpha_q_src=0.20,
    alpha_q_sink=-0.08,
    alpha_cop_src=0.08,
    alpha_cop_sink=-0.04,
)


def hp_capacity_kw(t_source_c, t_sink_c, params: QuasiSteadyHeatPumpParameters):
    return (
        params.q_ref_kw
        + params.alpha_q_src * (t_source_c - params.t_source_ref_c)
        + params.alpha_q_sink * (t_sink_c - params.t_sink_ref_c)
    )


def hp_cop(t_source_c, t_sink_c, params: QuasiSteadyHeatPumpParameters):
    return (
        params.cop_ref
        + params.alpha_cop_src * (t_source_c - params.t_source_ref_c)
        + params.alpha_cop_sink * (t_sink_c - params.t_sink_ref_c)
    )


# Quick sanity check
print("Example HP operating point")
print("Q_max [kW]:", round(float(hp_capacity_kw(5.0, 40.0, HP_PARAMS)), 3))
print("COP [-]:", round(float(hp_cop(5.0, 40.0, HP_PARAMS)), 3))


## Tank model

For the first integrated version we use a **3-node stratified tank** everywhere:
- in the controller,
- and in the plant simulation.

To keep the first build robust, the control-oriented formulation is written in **power form**:
- the heat pump charges the **top node**,
- the building extracts useful heat from the **top node**,
- vertical energy exchange between nodes is represented by a simple conductance,
- losses to the ambient are included.

This is intentionally simpler than a full inlet-mixing model, but still allows the controller to exploit stratification.


In [ ]:
# -----------------------------
# 3-node tank model
# -----------------------------
@dataclass
class ThreeNodeTankParameters:
    volume_m3: float
    ua_total_kw_per_k: float
    g_between_nodes_kw_per_k: float
    t_min_c: float
    t_max_c: float
    rho_kg_per_m3: float = 1000.0
    cp_kwh_per_kgk: float = 4180.0 / 3600.0 / 1000.0


class ThreeNodeTankModel:
    def __init__(self, params: ThreeNodeTankParameters, dt_hours: float) -> None:
        self.params = params
        self.dt_hours = dt_hours

        c_total = params.rho_kg_per_m3 * params.volume_m3 * params.cp_kwh_per_kgk
        c_node = c_total / 3.0
        ua_node = params.ua_total_kw_per_k / 3.0
        g = params.g_between_nodes_kw_per_k

        # State ordering: [bottom, middle, top]
        A_c = np.array([
            [-(g + ua_node) / c_node, g / c_node, 0.0],
            [g / c_node, -(2.0 * g + ua_node) / c_node, g / c_node],
            [0.0, g / c_node, -(g + ua_node) / c_node],
        ])

        # Inputs: [Q_hp, Q_build, T_amb]
        B_c = np.array([
            [0.0, 0.0, ua_node / c_node],
            [0.0, 0.0, ua_node / c_node],
            [1.0 / c_node, -1.0 / c_node, ua_node / c_node],
        ])

        self.A_d, self.B_d = discretize_linear_system(A_c, B_c, dt_hours)

    def step_numpy(self, x_k: np.ndarray, q_hp_kw: float, q_build_kw: float, t_amb_c: float) -> np.ndarray:
        u_k = np.array([q_hp_kw, q_build_kw, t_amb_c])
        x_next = self.A_d @ x_k + self.B_d @ u_k
        return np.clip(x_next, self.params.t_min_c, self.params.t_max_c)


TANK_PARAMS = ThreeNodeTankParameters(
    volume_m3=TANK_VOLUME_M3,
    ua_total_kw_per_k=0.002,        # 2 W/K
    g_between_nodes_kw_per_k=0.08,  # simple inter-node conductance
    t_min_c=25.0,
    t_max_c=60.0,
)

tank_model = ThreeNodeTankModel(TANK_PARAMS, dt_hours=DT_HOURS)

print(f"Tank volume [m³]: {TANK_VOLUME_M3:.3f}")
print("Discrete tank A matrix:")
print(np.round(tank_model.A_d, 4))
print("\nDiscrete tank B matrix:")
print(np.round(tank_model.B_d, 4))


## Economic MPC formulation

The controller uses:
- **building states**: indoor and envelope temperatures,
- **tank states**: bottom, middle, top temperatures,
- **decision variables**:
  - `Q_hp[k]`: thermal power produced by the heat pump,
  - `Q_build[k]`: thermal power delivered to the building from the tank.

### Objective
At each step, the MPC minimizes:
1. electricity cost of the heat pump,
2. indoor temperature tracking penalty,
3. comfort slack penalties,
4. small smoothing penalties on control variations.

### Important simplification
The RC building model is preserved; the MPC is made storage-sensitive by:
- introducing the tank states directly in the prediction model,
- linking heat pump capacity and COP to the **top tank temperature**,
- using the **time-band electricity price** in the objective.


In [ ]:
# -----------------------------
# MPC builder
# -----------------------------
def build_economic_mpc_solver(
    building_model: BuildingRCModel,
    tank_model: ThreeNodeTankModel,
    hp_params: QuasiSteadyHeatPumpParameters,
    horizon_steps: int,
):
    opti = ca.Opti()

    # States
    X_build = opti.variable(2, horizon_steps + 1)  # [Ti, Te]
    X_tank = opti.variable(3, horizon_steps + 1)   # [T_bottom, T_middle, T_top]

    # Controls
    Q_hp = opti.variable(1, horizon_steps)
    Q_build = opti.variable(1, horizon_steps)

    # Soft comfort constraints
    eps_low = opti.variable(1, horizon_steps)
    eps_high = opti.variable(1, horizon_steps)

    # Parameters / forecasts
    x0_build = opti.parameter(2)
    x0_tank = opti.parameter(3)
    ta_forecast = opti.parameter(1, horizon_steps)
    gh_forecast = opti.parameter(1, horizon_steps)
    price_forecast = opti.parameter(1, horizon_steps)

    opti.subject_to(X_build[:, 0] == x0_build)
    opti.subject_to(X_tank[:, 0] == x0_tank)

    objective = 0

    A_b = ca.DM(building_model.A_d)
    B_b = ca.DM(building_model.B_d)
    A_t = ca.DM(tank_model.A_d)
    B_t = ca.DM(tank_model.B_d)

    for k in range(horizon_steps):
        # Building dynamics: same 3R2C structure, with Q_build as delivered heat input
        u_build = ca.vertcat(
            Q_build[0, k] * building_model.params.heating_efficiency,
            ta_forecast[0, k],
            gh_forecast[0, k],
        )
        x_build_next = A_b @ X_build[:, k] + B_b @ u_build
        opti.subject_to(X_build[:, k + 1] == x_build_next)

        # Tank dynamics
        u_tank = ca.vertcat(
            Q_hp[0, k],
            Q_build[0, k],
            ta_forecast[0, k],
        )
        x_tank_next = A_t @ X_tank[:, k] + B_t @ u_tank
        opti.subject_to(X_tank[:, k + 1] == x_tank_next)

        # HP algebraic maps
        t_top = X_tank[2, k]
        q_hp_max = hp_capacity_kw(ta_forecast[0, k], t_top, hp_params)
        cop_k = hp_cop(ta_forecast[0, k], t_top, hp_params)

        # Control bounds
        opti.subject_to(Q_hp[0, k] >= 0.0)
        opti.subject_to(Q_build[0, k] >= 0.0)
        opti.subject_to(Q_build[0, k] <= 6.0)
        opti.subject_to(Q_hp[0, k] <= q_hp_max)

        # State bounds
        opti.subject_to(X_tank[:, k] >= tank_model.params.t_min_c)
        opti.subject_to(X_tank[:, k] <= tank_model.params.t_max_c)

        # Comfort soft constraints
        opti.subject_to(eps_low[0, k] >= 0.0)
        opti.subject_to(eps_high[0, k] >= 0.0)
        opti.subject_to(X_build[0, k] + eps_low[0, k] >= TI_MIN_C)
        opti.subject_to(X_build[0, k] - eps_high[0, k] <= TI_MAX_C)

        # Stage cost
        p_hp_el_kw = Q_hp[0, k] / cop_k
        objective += price_forecast[0, k] * p_hp_el_kw * DT_HOURS
        objective += TRACK_WEIGHT * (X_build[0, k] - TI_REF_C) ** 2
        objective += SLACK_WEIGHT * (eps_low[0, k] ** 2 + eps_high[0, k] ** 2)

        if k > 0:
            objective += SMOOTH_WEIGHT * (
                (Q_hp[0, k] - Q_hp[0, k - 1]) ** 2
                + (Q_build[0, k] - Q_build[0, k - 1]) ** 2
            )

    # Terminal bounds
    opti.subject_to(X_tank[:, -1] >= tank_model.params.t_min_c)
    opti.subject_to(X_tank[:, -1] <= tank_model.params.t_max_c)

    opti.minimize(objective)
    opti.solver(
        "ipopt",
        {"print_time": False},
        {"print_level": 0, "max_iter": 250},
    )

    return opti, {
        "X_build": X_build,
        "X_tank": X_tank,
        "Q_hp": Q_hp,
        "Q_build": Q_build,
        "eps_low": eps_low,
        "eps_high": eps_high,
        "x0_build": x0_build,
        "x0_tank": x0_tank,
        "ta_forecast": ta_forecast,
        "gh_forecast": gh_forecast,
        "price_forecast": price_forecast,
    }


opti, mpc_vars = build_economic_mpc_solver(
    building_model=building_model,
    tank_model=tank_model,
    hp_params=HP_PARAMS,
    horizon_steps=HORIZON_STEPS,
)

print(f"MPC horizon steps: {HORIZON_STEPS}")


In [ ]:
# -----------------------------
# Closed-loop simulation
# -----------------------------
def run_closed_loop_simulation(
    df: pd.DataFrame,
    building_model: BuildingRCModel,
    tank_model: ThreeNodeTankModel,
    hp_params: QuasiSteadyHeatPumpParameters,
    opti,
    mpc_vars,
):
    # Initial conditions
    x_build = np.array([21.5, 21.5], dtype=float)      # [Ti, Te]
    x_tank = np.array([35.0, 40.0, 45.0], dtype=float) # [bottom, middle, top]

    records = []

    n_steps = len(df) - 1 - HORIZON_STEPS
    for k in range(n_steps):
        horizon_slice = slice(k, k + HORIZON_STEPS)

        ta_fc = df["Ta"].iloc[horizon_slice].to_numpy(dtype=float).reshape(1, -1)
        gh_fc = df["Gv"].iloc[horizon_slice].to_numpy(dtype=float).reshape(1, -1)
        price_fc = df["price_eur_per_kwh"].iloc[horizon_slice].to_numpy(dtype=float).reshape(1, -1)

        opti.set_value(mpc_vars["x0_build"], x_build)
        opti.set_value(mpc_vars["x0_tank"], x_tank)
        opti.set_value(mpc_vars["ta_forecast"], ta_fc)
        opti.set_value(mpc_vars["gh_forecast"], gh_fc)
        opti.set_value(mpc_vars["price_forecast"], price_fc)

        try:
            solution = opti.solve()
            q_hp_cmd = float(solution.value(mpc_vars["Q_hp"][0, 0]))
            q_build_cmd = float(solution.value(mpc_vars["Q_build"][0, 0]))
        except Exception as exc:
            print(f"Optimization failed at step {k}: {exc}")
            break

        # Plant update
        ta_k = float(df["Ta"].iloc[k])
        gh_k = float(df["Gv"].iloc[k])
        price_k = float(df["price_eur_per_kwh"].iloc[k])

        x_build = building_model.step_numpy(
            x_k=x_build,
            q_build_kw=q_build_cmd,
            t_out_c=ta_k,
            g_solar=gh_k,
        )

        x_tank = tank_model.step_numpy(
            x_k=x_tank,
            q_hp_kw=q_hp_cmd,
            q_build_kw=q_build_cmd,
            t_amb_c=ta_k,
        )

        # Realized HP electricity use
        q_hp_max_real = float(hp_capacity_kw(ta_k, x_tank[2], hp_params))
        cop_real = float(hp_cop(ta_k, x_tank[2], hp_params))
        p_hp_real_kw = q_hp_cmd / max(cop_real, 1.1)
        cost_step_eur = price_k * p_hp_real_kw * DT_HOURS

        records.append({
            "timestamp": df["timestamp"].iloc[k],
            "Ta_C": ta_k,
            "Gv": gh_k,
            "price_eur_per_kwh": price_k,
            "band": df["band"].iloc[k],
            "Q_hp_kw": q_hp_cmd,
            "Q_build_kw": q_build_cmd,
            "P_hp_kw": p_hp_real_kw,
            "Q_hp_max_kw": q_hp_max_real,
            "COP": cop_real,
            "Ti_C": float(x_build[0]),
            "Te_C": float(x_build[1]),
            "T_tank_bottom_C": float(x_tank[0]),
            "T_tank_middle_C": float(x_tank[1]),
            "T_tank_top_C": float(x_tank[2]),
            "cost_step_eur": cost_step_eur,
        })

    results = pd.DataFrame(records)
    if not results.empty:
        results["cumulative_cost_eur"] = results["cost_step_eur"].cumsum()
        results["cumulative_hp_energy_kwh"] = (results["P_hp_kw"] * DT_HOURS).cumsum()
    return results


results = run_closed_loop_simulation(
    df=df_inputs,
    building_model=building_model,
    tank_model=tank_model,
    hp_params=HP_PARAMS,
    opti=opti,
    mpc_vars=mpc_vars,
)

results.head()


In [ ]:
# -----------------------------
# Diagnostics and plots
# -----------------------------
if results.empty:
    raise RuntimeError("No results were produced. Check the optimization setup.")

print("Simulation summary")
print("-" * 40)
print(f"Time steps simulated: {len(results)}")
print(f"Indoor temperature range [°C]: {results['Ti_C'].min():.2f} -> {results['Ti_C'].max():.2f}")
print(f"Top tank temperature range [°C]: {results['T_tank_top_C'].min():.2f} -> {results['T_tank_top_C'].max():.2f}")
print(f"Total HP electricity use [kWh]: {results['P_hp_kw'].sum() * DT_HOURS:.2f}")
print(f"Total electricity cost [€]: {results['cost_step_eur'].sum():.2f}")

fig, axes = plt.subplots(5, 1, figsize=(12, 14), sharex=True)

axes[0].plot(results["timestamp"], results["Ti_C"], label="Indoor temperature")
axes[0].axhline(TI_MIN_C, color="k", linestyle="--", alpha=0.4, label="Comfort bounds")
axes[0].axhline(TI_MAX_C, color="k", linestyle="--", alpha=0.4)
axes[0].set_ylabel("Ti [°C]")
axes[0].legend()
axes[0].grid(True)

axes[1].step(results["timestamp"], results["Q_hp_kw"], where="post", label="HP thermal power")
axes[1].step(results["timestamp"], results["Q_build_kw"], where="post", label="Delivered heat to building")
axes[1].set_ylabel("Power [kW]")
axes[1].legend()
axes[1].grid(True)

axes[2].plot(results["timestamp"], results["T_tank_bottom_C"], label="Tank bottom")
axes[2].plot(results["timestamp"], results["T_tank_middle_C"], label="Tank middle")
axes[2].plot(results["timestamp"], results["T_tank_top_C"], label="Tank top")
axes[2].set_ylabel("Tank T [°C]")
axes[2].legend()
axes[2].grid(True)

axes[3].step(results["timestamp"], results["price_eur_per_kwh"], where="post", label="Electricity price")
axes[3].set_ylabel("€/kWh")
axes[3].legend()
axes[3].grid(True)

axes[4].plot(results["timestamp"], results["cumulative_cost_eur"], label="Cumulative cost")
axes[4].set_ylabel("€")
axes[4].set_xlabel("Time")
axes[4].legend()
axes[4].grid(True)

plt.tight_layout()
plt.show()


## Remarks for the next iteration

This v1 is intentionally conservative and readable.

### What is already in place
- the **building RC structure** is preserved,
- the **MPC is storage-sensitive**,
- the **electricity tariff signal** is active,
- the **HP and tank communicate consistently**,
- the **economic objective** is already meaningful.

### Natural next upgrades
1. Move from **3-node everywhere** to:
   - 3 nodes in the controller,
   - 5 nodes in the plant.
2. Add a more explicit relation between:
   - tank top temperature,
   - supply temperature,
   - useful extractable heat.
3. Replace constant yearly band prices with:
   - monthly values,
   - or measured tariff series.
4. Add optional direct **HP → building** bypass if needed in a later architecture.
